In [51]:
import pandas as pd
import geopandas as gpd
import os

In [52]:
epa = gpd.read_file("/work/hawkins_lab/vulcan/data/gis/epa-sld/smartlocationdb_4326.shp", engine="pyogrio", use_arrow=True)
# Drop the geometry column
epa = epa.drop(columns='geometry')

epa_cbsa = pd.read_csv("/work/hawkins_lab/vulcan/data/transport/epa_cbsa.csv")
gdf = gpd.read_file('/work/hawkins_lab/vulcan/data/vulcan/scope2/ELC2.CO2.BG.ann.smplst.mn.2019.gpkg', engine="pyogrio", use_arrow=True)

In [53]:
# Replace column names using the dictionary
column_mapping = {'Block group ID':'geoid10','Res CO2 (tC)':'res_tc','Com CO2 (tC)':'com_tc','Ind CO2 (tC)':'ind_tc'}
gdf = gdf.rename(columns=column_mapping)

gdf.head()

,geoid10,res_tc,com_tc,ind_tc,geometry
0,010010201001,405.223297,509.023092,538.749232,"MULTIPOLYGON (((987112.673 -775319.167, 987182..."
1,010010201002,808.444740,74.663462,10.411046,"MULTIPOLYGON (((987182.576 -775338.645, 987112..."
2,010010202001,455.802904,582.339788,252.716984,"MULTIPOLYGON (((988641.227 -773190.54, 988620...."
3,010010202002,562.054501,2065.913506,230.887294,"MULTIPOLYGON (((988774.186 -774812.669, 988799..."
4,010010203001,1208.963080,400.449699,192.718242,"MULTIPOLYGON (((989044.883 -774778.916, 988989..."


In [54]:
# Convert all column names to lowercase
epa.columns = epa.columns.str.lower()
# Filter out anything where FIPS doesn't match the GEOID
epa['fips'] = epa.geoid20.str[:5].astype(int)

# Filter out anything outside the 50 states
epa = epa[((epa.fips/1000).astype(int) <=56)]
epa.loc[epa.cbsa.isna(),"cbsa"] = epa.loc[epa.cbsa.isna(),"statefp"].astype(int) + epa.loc[epa.cbsa.isna(),"countyfp"].astype(int)/100
epa["cbsa"] = round(epa["cbsa"].astype(float),2)

epa = pd.merge(epa, epa_cbsa, left_on='cbsa', right_on='cbsa_cbsa', how='left')

In [55]:
epa.head()

,objectid,geoid10,geoid20,statefp,countyfp,tractce,blkgrpce,csa,csa_name,cbsa,...,d3aao_cbsa,d3amm_cbsa,d3apo_cbsa,d3b_cbsa,d3bao_cbsa,d3bmm3_cbsa,d3bmm4_cbsa,d3bpo3_cbsa,d3bpo4_cbsa,d4d_cbsa
0,1.0,481130078254,481130078254,48,113,007825,4,206,"Dallas-Fort Worth, TX-OK",19100.0,...,0.221626,0.744148,0.246234,0.861672,0.275285,0.794435,0.04687,0.330416,0.064528,-99999.0
1,2.0,481130078252,481130078252,48,113,007825,2,206,"Dallas-Fort Worth, TX-OK",19100.0,...,0.221626,0.744148,0.246234,0.861672,0.275285,0.794435,0.04687,0.330416,0.064528,-99999.0
2,3.0,481130078253,481130078253,48,113,007825,3,206,"Dallas-Fort Worth, TX-OK",19100.0,...,0.221626,0.744148,0.246234,0.861672,0.275285,0.794435,0.04687,0.330416,0.064528,-99999.0
3,4.0,481130078241,481130078241,48,113,007824,1,206,"Dallas-Fort Worth, TX-OK",19100.0,...,0.221626,0.744148,0.246234,0.861672,0.275285,0.794435,0.04687,0.330416,0.064528,-99999.0
4,5.0,481130078242,481130078242,48,113,007824,2,206,"Dallas-Fort Worth, TX-OK",19100.0,...,0.221626,0.744148,0.246234,0.861672,0.275285,0.794435,0.04687,0.330416,0.064528,-99999.0


In [56]:
epa["statefp"] = epa.statefp.astype(int)
epa.statefp.max()

56

In [57]:
gdf10 = pd.merge(gdf, epa, on='geoid10', how='left')
gdf20 = pd.merge(gdf, epa, left_on='geoid10', right_on='geoid20', how='left').drop(columns=["geoid10_y"]).rename(columns={"geoid10_x":"geoid10"})
gdf = gdf10
gdf[gdf10.geoid20.isna()] = gdf20[gdf10.geoid20.isna()]

In [58]:
gdf.statefp.unique()

array([ 1.,  2.,  4.,  5.,  6.,  8.,  9., 10., 11., 12., 13., 15., 16.,
       17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 28., 29.,
       30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40., 41., 42.,
       44., 45., 46., 47., 48., 49., 50., 51., 53., 54., 55., 56.])

In [59]:
def get_id_attr(df, yr, months, suffix):
    df['ctyfips'] = df.ids.str[:5].astype(int)
    df['year'] = df.ids.str[-4:].astype(int)
    df = df[df.year==yr]
    df.columns = [ col + "_" + suffix if col in months else col
        for col in df.columns
    ]
    df = df.drop(columns=["ids"])
    df = df.set_index(['ctyfips', 'year'])
    return df

In [60]:
months = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

cols = ["ids"]+months

cty_crosswalk = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/county-to-climdivs.txt", sep='\s+')

df_precip = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-pcpncy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_precip["all"] = df_precip.loc[:, months].sum(axis=1)
df_temp_min = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-tmincy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_temp_min["all"] = df_temp_min.loc[:, months].min(axis=1)
df_temp_mean = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-tmpccy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_temp_mean["all"] = df_temp_mean.loc[:, months].mean(axis=1)
df_temp_max = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-tmaxcy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_temp_max["all"] = df_temp_max.loc[:, months].max(axis=1)
df_cdd = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-cddccy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_cdd["all"] = df_cdd.loc[:, months].sum(axis=1)
df_hdd = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-hddccy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_hdd["all"] = df_hdd.loc[:, months].sum(axis=1)

months = months+["all"]

# Process first column to extract location and year details - filter for Vulcan year of 2019
df_precip = get_id_attr(df_precip, 2019, months, "prec")
df_temp_min = get_id_attr(df_temp_min, 2019, months, "tmin")
df_temp_mean = get_id_attr(df_temp_mean, 2019, months, "tmean")
df_temp_max = get_id_attr(df_temp_max, 2019, months, "tmax")
df_cdd = get_id_attr(df_cdd, 2019, months, "cdd")
df_hdd = get_id_attr(df_hdd, 2019, months, "hdd")

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_958047/3718008754.py:5: SyntaxWarning: invalid escape sequence '\s'
  cty_crosswalk = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/county-to-climdivs.txt", sep='\s+')
/tmp/ipykernel_958047/3718008754.py:7: SyntaxWarning: invalid escape sequence '\s'
  df_precip = pd.read

In [61]:
# Join the dataframes into one
df_climate = df_precip.join(df_temp_min, on=['ctyfips', 'year'], how='left').join(df_temp_mean, on=['ctyfips', 'year'], how='left')\
            .join(df_temp_max, on=['ctyfips', 'year'], how='left').join(df_cdd, on=['ctyfips', 'year'], how='left').join(df_hdd, on=['ctyfips', 'year'], how='left').reset_index()

add_cbg_stats = pd.read_csv('/work/hawkins_lab/vulcan/data/nhgis0021_ds239_20185_blck_grp_clean.csv', usecols=['geoid','p_highschool','p_owner','avg_hh_size','med_dwelling_age'])
add_cbg_stats['geoid20'] = add_cbg_stats['geoid'].str[7:].astype(int)
gdf['geoid20'] = gdf['geoid20'].astype(int)

In [62]:
cty_crosswalk = dict(zip(cty_crosswalk.NCDC_FIPS_ID,cty_crosswalk.POSTAL_FIPS_ID))

In [63]:
df_climate.ctyfips = df_climate.ctyfips.map(cty_crosswalk)

In [64]:
df_fips = pd.read_csv('state_and_county_fips_master.csv')
df_climate = pd.merge(df_climate, df_fips, left_on="ctyfips",right_on="fips",how="right")

df_climate = df_climate.sort_values(by="fips", ascending=False)
df_climate = df_climate.bfill()
df_climate = df_climate.bfill()
df_climate.ctyfips = df_climate.fips

In [65]:
df_co2 = pd.read_csv('/work/hawkins_lab/vulcan/data/egrid_state_2019.csv', header=1)
df_co2.columns = df_co2.columns.str.lower()

In [66]:
gdf["statefp"] = gdf.statefp.astype(int)
gdf["countyfp"] = gdf.countyfp.astype(int)
gdf["ctyfips"] = gdf["statefp"]*1000 + gdf["countyfp"]

gdf = pd.merge(gdf,df_co2, left_on="statefp", right_on="fipsst", how="left")
gdf = pd.merge(gdf, df_climate, on="ctyfips", how="left")
gdf = pd.merge(gdf, add_cbg_stats, on="geoid20", how="left")

In [67]:
# Fill NaN in CO2 columns - confirmed these are parks, etc. without electricity (n=23)
gdf[['res_tc', 'com_tc', 'ind_tc']] = gdf[['res_tc', 'com_tc', 'ind_tc']].fillna(0)

In [68]:
par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_SC2_epa_climate.parquet'
gdf.to_parquet(par_file)

In [69]:
gdf.isna().sum()

geoid10             0
res_tc              0
com_tc              0
ind_tc              0
geometry            0
                   ..
geoid               0
p_highschool        0
p_owner             0
avg_hh_size         0
med_dwelling_age    0
Length: 383, dtype: int64